In [2]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm

db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Compustat Query - Get R&D Data and Link to CRSP

In [3]:
comp_sql = """
SELECT
    a.gvkey, 
    a.datadate, 
    a.xrd,
    b.lpermno AS permno
FROM
    comp.funda a
JOIN
    crsp.ccmxpf_linktable b ON a.gvkey = b.gvkey
WHERE
    a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
    AND a.fyear BETWEEN 1978 AND 2022
    AND a.fic = 'USA'
    AND a.curcd = 'USD'
    AND a.exchg BETWEEN 11 AND 19
    AND (a.sich < 6000 OR a.sich > 6999 OR a.sich IS NULL)
    AND (a.sich != 2834 OR a.sich IS NULL)
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C')
    AND b.usedflag = 1
    AND (b.linkdt <= a.datadate OR b.linkdt IS NULL)
    AND (b.linkenddt >= a.datadate OR b.linkenddt IS NULL)
"""
comp = db.raw_sql(comp_sql)

## CRSP Query - Monthly Returns with Delisting Adjustment

In [4]:
crsp_sql = """
SELECT 
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    c.dlret
FROM 
    crsp.msf a
INNER JOIN 
    crsp.msenames b
ON 
    a.permno = b.permno 
    AND a.date BETWEEN b.namedt AND b.nameendt
LEFT JOIN
    crsp.msedelist c
ON
    a.permno = c.permno
    AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE 
    a.date BETWEEN '1980-01-01' AND '2023-06-30'
    AND a.ret IS NOT NULL
    AND a.ret >= -1
    AND b.shrcd IN (10, 11)
    AND (b.siccd < 6000 OR b.siccd > 6999)
    AND (b.siccd != 2834)
    AND b.exchcd IN (1, 2, 3)
"""
crsp = db.raw_sql(crsp_sql)

## Adjust for Delisting Returns

In [5]:
crsp['dlret'] = pd.to_numeric(crsp['dlret'], errors='coerce')
crsp['ret'] = np.where(crsp['dlret'].notna(),
                       (1 + crsp['ret']) * (1 + crsp['dlret']) - 1,
                       crsp['ret'])

## Market Return and Risk-Free Rate

In [6]:
mkt = db.raw_sql("""
    SELECT date, vwretd AS mkt_ret
    FROM crsp.msi
    WHERE date BETWEEN '1980-01-01' AND '2023-06-30'
""")

rf = db.raw_sql("""
    SELECT date, rf
    FROM ff.factors_monthly
    WHERE date BETWEEN '1980-01-01' AND '2023-06-30'
""")

## Create R&D Flag and Lag It

In [7]:
comp['xrd'] = pd.to_numeric(comp['xrd'], errors='coerce')
comp['has_rd'] = ((comp['xrd'].notna()) & (comp['xrd'] > 0)).astype(int)
comp = comp.sort_values(['permno', 'datadate'])
comp['rd_flag_lag1'] = comp.groupby('permno')['has_rd'].shift(1)

## Portfolio Year Assignment (Fama-French Timing)

In [8]:
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['portfolio_year'] = comp['datadate'].dt.year + 1

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['month'] = crsp['date'].dt.month
crsp['year'] = crsp['date'].dt.year
crsp['portfolio_year'] = np.where(crsp['month'] >= 7, crsp['year'], crsp['year'] - 1)

## Market Cap - Use Lagged for Value Weighting

In [9]:
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')
crsp['prc'] = pd.to_numeric(crsp['prc'], errors='coerce')
crsp['shrout'] = pd.to_numeric(crsp['shrout'], errors='coerce')
crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']
crsp = crsp.sort_values(['permno', 'date'])
crsp['mktcap_lag'] = crsp.groupby('permno')['mktcap'].shift(1)

## Merge Datasets

In [10]:
comp_for_merge = comp[['permno', 'portfolio_year', 'rd_flag_lag1']].copy()
comp_for_merge = comp_for_merge.drop_duplicates(['permno', 'portfolio_year'])
comp_for_merge = comp_for_merge.dropna(subset=['rd_flag_lag1'])

merged = crsp.merge(comp_for_merge, on=['permno', 'portfolio_year'], how='inner')
merged = merged[(merged['portfolio_year'] >= 1980) & (merged['portfolio_year'] <= 2022)]
merged = merged.dropna(subset=['ret', 'mktcap_lag'])
merged = merged[merged['mktcap_lag'] > 0]

## Calculate Portfolio Returns Each Month

In [11]:
merged['year_month'] = merged['date'].dt.to_period('M')

portfolio_returns = []
for (ym, rd), group in merged.groupby(['year_month', 'rd_flag_lag1']):
    ew_ret = group['ret'].mean()
    weights = group['mktcap_lag'] / group['mktcap_lag'].sum()
    vw_ret = (weights * group['ret']).sum()
    portfolio_returns.append({
        'date': group['date'].iloc[0],
        'rd_portfolio': int(rd),
        'ew_ret': ew_ret,
        'vw_ret': vw_ret
    })
portfolio_returns = pd.DataFrame(portfolio_returns)

## Merge in Market and Risk-Free Rate

In [12]:
portfolio_returns['date'] = pd.to_datetime(portfolio_returns['date'])
mkt['date'] = pd.to_datetime(mkt['date'])
rf['date'] = pd.to_datetime(rf['date'])
portfolio_returns = portfolio_returns.merge(mkt, on='date', how='left')
portfolio_returns = portfolio_returns.merge(rf, on='date', how='left')

portfolio_returns['ew_ret'] = pd.to_numeric(portfolio_returns['ew_ret'], errors='coerce')
portfolio_returns['vw_ret'] = pd.to_numeric(portfolio_returns['vw_ret'], errors='coerce')
portfolio_returns['mkt_ret'] = pd.to_numeric(portfolio_returns['mkt_ret'], errors='coerce')
portfolio_returns['rf'] = pd.to_numeric(portfolio_returns['rf'], errors='coerce').fillna(0)

## CAPM Regression Function

In [13]:
## CAPM Regression Function
def run_capm(data, ret_col):
    data = data.dropna(subset=[ret_col, 'mkt_ret', 'rf'])
    y = (data[ret_col] - data['rf']).astype(float).values
    x = (data['mkt_ret'] - data['rf']).astype(float).values
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    return model.params[0], model.tvalues[0]

## Run Regressions

In [14]:
rd_data = portfolio_returns[portfolio_returns['rd_portfolio'] == 1].copy()
no_rd_data = portfolio_returns[portfolio_returns['rd_portfolio'] == 0].copy()

rd_ew_alpha, rd_ew_t = run_capm(rd_data, 'ew_ret')
rd_vw_alpha, rd_vw_t = run_capm(rd_data, 'vw_ret')
no_rd_ew_alpha, no_rd_ew_t = run_capm(no_rd_data, 'ew_ret')
no_rd_vw_alpha, no_rd_vw_t = run_capm(no_rd_data, 'vw_ret')

## Results

In [15]:
print(f"{'Portfolio':<20} {'Alpha (Monthly)':<18} {'Alpha (Annual)':<18} {'t-stat':<10}")
print("-" * 70)
print(f"{'R&D (Equal-Wt)':<20} {rd_ew_alpha*100:>6.3f}%{'':11} {rd_ew_alpha*12*100:>6.2f}%{'':11} {rd_ew_t:>6.2f}")
print(f"{'R&D (Value-Wt)':<20} {rd_vw_alpha*100:>6.3f}%{'':11} {rd_vw_alpha*12*100:>6.2f}%{'':11} {rd_vw_t:>6.2f}")
print(f"{'No R&D (Equal-Wt)':<20} {no_rd_ew_alpha*100:>6.3f}%{'':11} {no_rd_ew_alpha*12*100:>6.2f}%{'':11} {no_rd_ew_t:>6.2f}")
print(f"{'No R&D (Value-Wt)':<20} {no_rd_vw_alpha*100:>6.3f}%{'':11} {no_rd_vw_alpha*12*100:>6.2f}%{'':11} {no_rd_vw_t:>6.2f}")

db.close()

Portfolio            Alpha (Monthly)    Alpha (Annual)     t-stat    
----------------------------------------------------------------------
R&D (Equal-Wt)        0.134%              1.61%              0.77
R&D (Value-Wt)        0.037%              0.45%              0.61
No R&D (Equal-Wt)     0.123%              1.48%              0.91
No R&D (Value-Wt)     0.068%              0.82%              1.15
